# OpenAI APIs - 补全

SGLang 提供 OpenAI 兼容的 API，以便从 OpenAI 服务平滑过渡到自托管的本地模型。
完整的 API 参考请查看 [OpenAI API 参考文档](https://platform.openai.com/docs/api-reference)。

本教程涵盖以下常用 API：

- `chat/completions`
- `completions`

查看其他教程以了解用于视觉语言模型的[视觉 API](openai_api_vision.ipynb) 和用于嵌入模型的[嵌入 API](openai_api_embeddings.ipynb)。

## 启动服务器

在终端中启动服务器并等待初始化完成。

In [ ]:
from sglang.test.doc_patch import launch_server_cmd
from sglang.utils import wait_for_server, print_highlight, terminate_process

server_process, port = launch_server_cmd(
    "python3 -m sglang.launch_server --model-path qwen/qwen2.5-0.5b-instruct --host 0.0.0.0 --log-level warning"
)

wait_for_server(f"http://localhost:{port}", process=server_process)
print(f"Server started on http://localhost:{port}")

## 聊天补全

### 用法

服务器完整实现了 OpenAI API。
它会自动应用 Hugging Face tokenizer 中指定的聊天模板（如果可用）。
您也可以在启动服务器时使用 `--chat-template` 指定自定义聊天模板。

In [ ]:
import openai

client = openai.Client(base_url=f"http://127.0.0.1:{port}/v1", api_key="None")

response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[
        {"role": "user", "content": "List 3 countries and their capitals."},
    ],
    temperature=0,
    max_tokens=64,
)

print_highlight(f"Response: {response}")

### 模型思考/推理支持

某些模型支持内部推理或思考过程，可以在 API 响应中展示。SGLang 通过 `chat_template_kwargs` 参数和兼容的推理解析器为各种推理模型提供统一支持。

#### 支持的模型和配置

| 模型系列 | 聊天模板参数 | 推理解析器 | 备注 |
|----------|-------------|-----------|------|
| DeepSeek-R1（R1、R1-0528、R1-Distill） | `enable_thinking` | `--reasoning-parser deepseek-r1` | 标准推理模型 |
| DeepSeek-V3.1 | `thinking` | `--reasoning-parser deepseek-v3` | 混合模型（支持思考/非思考模式） |
| Qwen3（标准版） | `enable_thinking` | `--reasoning-parser qwen3` | 混合模型（支持思考/非思考模式） |
| Qwen3-Thinking | 不适用（始终启用） | `--reasoning-parser qwen3-thinking` | 始终生成推理内容 |
| Kimi | 不适用（始终启用） | `--reasoning-parser kimi` | Kimi 思考模型 |
| Gpt-Oss | 不适用（始终启用） | `--reasoning-parser gpt-oss` | Gpt-Oss 思考模型 |

#### 基本用法

要启用推理输出，您需要：
1. 使用适当的推理解析器启动服务器
2. 在 `chat_template_kwargs` 中设置特定于模型的参数
3. 可选择使用 `separate_reasoning: False` 来不单独获取推理内容（默认为 `True`）

**Qwen3-Thinking 模型注意事项：** 这些模型始终生成思考内容，不支持 `enable_thinking` 参数。请使用 `--reasoning-parser qwen3-thinking` 或 `--reasoning-parser qwen3` 来解析思考内容。


#### 示例：Qwen3 模型

```python
# 启动服务器：
# python3 -m sglang.launch_server --model Qwen/Qwen3-4B --reasoning-parser qwen3

from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=f"http://127.0.0.1:30000/v1",
)

model = "Qwen/Qwen3-4B"
messages = [{"role": "user", "content": "How many r's are in 'strawberry'?"}]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    extra_body={
        "chat_template_kwargs": {"enable_thinking": True},
        "separate_reasoning": True
    }
)

print("Reasoning:", response.choices[0].message.reasoning_content)
print("-"*100)
print("Answer:", response.choices[0].message.content)
```

**示例输出：**
```
Reasoning: Okay, so the user is asking how many 'r's are in the word 'strawberry'. Let me think. First, I need to make sure I have the word spelled correctly. Strawberry... S-T-R-A-W-B-E-R-R-Y. Wait, is that right? Let me break it down.

Starting with 'strawberry', let's write out the letters one by one. S, T, R, A, W, B, E, R, R, Y. Hmm, wait, that's 10 letters. Let me check again. S (1), T (2), R (3), A (4), W (5), B (6), E (7), R (8), R (9), Y (10). So the letters are S-T-R-A-W-B-E-R-R-Y. 
...
Therefore, the answer should be three R's in 'strawberry'. But I need to make sure I'm not counting any other letters as R. Let me check again. S, T, R, A, W, B, E, R, R, Y. No other R's. So three in total. Yeah, that seems right.

----------------------------------------------------------------------------------------------------
Answer: The word "strawberry" contains **three** letters 'r'. Here's the breakdown:

1. **S-T-R-A-W-B-E-R-R-Y**  
   - The **third letter** is 'R'.  
   - The **eighth and ninth letters** are also 'R's.  

Thus, the total count is **3**.  

**Answer:** 3.
```

**注意：** 设置 `"enable_thinking": False`（或省略此参数）将导致 `reasoning_content` 为 `None`。Qwen3-Thinking 模型始终生成推理内容，不支持 `enable_thinking` 参数。


#### Logit Bias 支持

SGLang 在聊天补全和补全 API 中都支持 `logit_bias` 参数。该参数允许您通过向特定 token 的 logits 添加偏置值来修改其生成概率。偏置值范围为 -100 到 100，其中：

- **正值**（0 到 100）增加该 token 被选中的概率
- **负值**（-100 到 0）降低该 token 被选中的概率
- **-100** 实际上会阻止该 token 被生成

`logit_bias` 参数接受一个字典，其中键是 token ID（字符串形式），值是偏置量（浮点数）。


#### 获取 Token IDs

要有效使用 `logit_bias`，您需要知道要偏置的词的 token ID。以下是获取 token ID 的方法：

```python
# 获取 tokenizer 以查找 token ID
import tiktoken

# 对于 OpenAI 模型，使用相应的编码
tokenizer = tiktoken.encoding_for_model("gpt-3.5-turbo")  # 或您的模型

# 获取特定词的 token ID
word = "sunny"
token_ids = tokenizer.encode(word)
print(f"Token IDs for '{word}': {token_ids}")

# 对于 SGLang 模型，您可以通过客户端访问 tokenizer
# 并获取用于偏置的 token ID
```

**重要：** `logit_bias` 参数使用 token ID 作为字符串键，而不是实际的词。


#### 示例：DeepSeek-V3 模型

DeepSeek-V3 模型通过 `thinking` 参数支持思考模式：

```python
# 启动服务器：
# python3 -m sglang.launch_server --model deepseek-ai/DeepSeek-V3.1 --tp 8  --reasoning-parser deepseek-v3

from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url=f"http://127.0.0.1:30000/v1",
)

model = "deepseek-ai/DeepSeek-V3.1"
messages = [{"role": "user", "content": "How many r's are in 'strawberry'?"}]

response = client.chat.completions.create(
    model=model,
    messages=messages,
    extra_body={
        "chat_template_kwargs": {"thinking": True},
        "separate_reasoning": True
    }
)

print("Reasoning:", response.choices[0].message.reasoning_content)
print("-"*100)
print("Answer:", response.choices[0].message.content)
```

**示例输出：**
```
Reasoning: First, the question is: "How many r's are in 'strawberry'?"

I need to count the number of times the letter 'r' appears in the word "strawberry".

Let me write out the word: S-T-R-A-W-B-E-R-R-Y.

Now, I'll go through each letter and count the 'r's.
...
So, I have three 'r's in "strawberry".

I should double-check. The word is spelled S-T-R-A-W-B-E-R-R-Y. The letters are at positions: 3, 8, and 9 are 'r's. Yes, that's correct.

Therefore, the answer should be 3.
----------------------------------------------------------------------------------------------------
Answer: The word "strawberry" contains **3** instances of the letter "r". Here's a breakdown for clarity:

- The word is spelled: S-T-R-A-W-B-E-R-R-Y
- The "r" appears at the 3rd, 8th, and 9th positions.
```

**注意：** DeepSeek-V3 模型使用 `thinking` 参数（而非 `enable_thinking`）来控制推理输出。


In [ ]:
# Example with logit_bias parameter
# Note: You need to get the actual token IDs from your tokenizer
# For demonstration, we'll use some example token IDs
response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[
        {"role": "user", "content": "Complete this sentence: The weather today is"}
    ],
    temperature=0.7,
    max_tokens=20,
    logit_bias={
        "12345": 50,  # Increase likelihood of token ID 12345
        "67890": -50,  # Decrease likelihood of token ID 67890
        "11111": 25,  # Slightly increase likelihood of token ID 11111
    },
)

print_highlight(f"Response with logit bias: {response.choices[0].message.content}")

### 参数

聊天补全 API 接受 OpenAI Chat Completions API 的参数。详情请参阅 [OpenAI Chat Completions API](https://platform.openai.com/docs/api-reference/chat/create)。

SGLang 通过 `extra_body` 参数扩展了标准 API，允许进行额外的自定义。`extra_body` 中的一个关键选项是 `chat_template_kwargs`，可用于向聊天模板处理器传递参数。

In [ ]:
response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[
        {
            "role": "system",
            "content": "You are a knowledgeable historian who provides concise responses.",
        },
        {"role": "user", "content": "Tell me about ancient Rome"},
        {
            "role": "assistant",
            "content": "Ancient Rome was a civilization centered in Italy.",
        },
        {"role": "user", "content": "What were their major achievements?"},
    ],
    temperature=0.3,  # Lower temperature for more focused responses
    max_tokens=128,  # Reasonable length for a concise response
    top_p=0.95,  # Slightly higher for better fluency
    presence_penalty=0.2,  # Mild penalty to avoid repetition
    frequency_penalty=0.2,  # Mild penalty for more natural language
    n=1,  # Single response is usually more stable
    seed=42,  # Keep for reproducibility
)

print_highlight(response.choices[0].message.content)

流式模式也是支持的。

#### Logit Bias 支持

补全 API 也支持 `logit_bias` 参数，功能与上述聊天补全部分描述的相同。


In [ ]:
stream = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[{"role": "user", "content": "Say this is a test"}],
    stream=True,
)
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

#### 返回路由专家（MoE 模型）

对于 MoE 模型，在 `extra_body` 中设置 `return_routed_experts: true` 可返回专家路由数据。需要服务器启动时添加 `--enable-return-routed-experts` 标志。`routed_experts` 字段将在每个 choice 的 `sgl_ext` 对象中返回，包含 base64 编码的 int32 专家 ID，作为逻辑形状为 `[num_tokens, num_layers, top_k]` 的扁平化数组。

In [ ]:
# Example with logit_bias parameter for completions API
# Note: You need to get the actual token IDs from your tokenizer
# For demonstration, we'll use some example token IDs
response = client.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    prompt="The best programming language for AI is",
    temperature=0.7,
    max_tokens=20,
    logit_bias={
        "12345": 75,  # Strongly favor token ID 12345
        "67890": -100,  # Completely avoid token ID 67890
        "11111": -25,  # Slightly discourage token ID 11111
    },
)

print_highlight(f"Response with logit bias: {response.choices[0].text}")

## 补全

### 用法
补全 API 与聊天补全 API 类似，但没有 `messages` 参数或聊天模板。

In [ ]:
response = client.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    prompt="List 3 countries and their capitals.",
    temperature=0,
    max_tokens=64,
    n=1,
    stop=None,
)

print_highlight(f"Response: {response}")

### 参数

补全 API 接受 OpenAI Completions API 的参数。详情请参阅 [OpenAI Completions API](https://platform.openai.com/docs/api-reference/completions/create)。

以下是详细补全请求的示例：

In [ ]:
response = client.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    prompt="Write a short story about a space explorer.",
    temperature=0.7,  # Moderate temperature for creative writing
    max_tokens=150,  # Longer response for a story
    top_p=0.9,  # Balanced diversity in word choice
    stop=["\n\n", "THE END"],  # Multiple stop sequences
    presence_penalty=0.3,  # Encourage novel elements
    frequency_penalty=0.3,  # Reduce repetitive phrases
    n=1,  # Generate one completion
    seed=123,  # For reproducible results
)

print_highlight(f"Response: {response}")

#### 返回路由专家（MoE 模型）

对于 MoE 模型，在 `extra_body` 中设置 `return_routed_experts: true` 可返回专家路由数据。需要服务器启动时添加 `--enable-return-routed-experts` 标志。`routed_experts` 字段将在每个 choice 的 `sgl_ext` 对象中返回，包含 base64 编码的 int32 专家 ID，作为逻辑形状为 `[num_tokens, num_layers, top_k]` 的扁平化数组。

## 结构化输出（JSON、Regex、EBNF）

有关 OpenAI 兼容的结构化输出 API，请参阅[结构化输出](../advanced_features/structured_outputs.ipynb)了解更多详情。


## 使用 LoRA 适配器

SGLang 支持通过 OpenAI 兼容 API 使用 LoRA（低秩自适应）适配器。您可以直接在 `model` 参数中使用 `base-model:adapter-name` 语法指定要使用的适配器。

**服务器设置：**
```bash
python -m sglang.launch_server \
    --model-path qwen/qwen2.5-0.5b-instruct \
    --enable-lora \
    --lora-paths adapter_a=/path/to/adapter_a adapter_b=/path/to/adapter_b
```

有关 LoRA 服务配置的更多详情，请参阅 [LoRA 文档](../advanced_features/lora.ipynb)。

**API 调用：**

（推荐）使用 `model:adapter` 语法指定要使用的适配器：
```python
response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct:adapter_a",  # ← base-model:adapter-name
    messages=[{"role": "user", "content": "Convert to SQL: show all users"}],
    max_tokens=50,
)
```

**向后兼容：使用 `extra_body`**

旧的 `extra_body` 方法仍然支持，以保持向后兼容性：
```python
# 向后兼容方法
response = client.chat.completions.create(
    model="qwen/qwen2.5-0.5b-instruct",
    messages=[{"role": "user", "content": "Convert to SQL: show all users"}],
    extra_body={"lora_path": "adapter_a"},  # ← 旧方法
    max_tokens=50,
)
```
**注意：** 当同时指定 `model:adapter` 和 `extra_body["lora_path"]` 时，`model:adapter` 语法优先。

In [ ]:
terminate_process(server_process)